# 01 — The three datasets

Norman 2019, Adamson 2016 and Dixit 2016, all public and all reached through GEARS' own `PertData`
loader so that the preprocessing is the published one rather than ours. Nothing is committed to this
repository: the files land in `~/.cache/pertreadout/` and `data/checksums.json` records what should
have arrived.

The point of this notebook is to establish two facts that the rest of the analysis rests on:

1. **Only Norman has two-gene combinations.** Adamson and Dixit are single-gene screens, so
   "generalisation to novel perturbation *pairs*" is testable on Norman alone. The other two test
   generalisation to unseen single perturbations.
2. **Most genes do not measurably move.** That is what makes genome-wide MSE small, and it is what
   makes a sign-based metric scored over all genes uninterpretable — see `04`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

RUNS = REPO / "runs"
TABLES = REPO / "results" / "tables"
pd.set_option("display.width", 160)

In [ ]:
from pertreadout import data as dm

print(dm.verify_manifest() if dm.MANIFEST_PATH.exists() else "no manifest committed yet")

In [ ]:
import scanpy as sc

rows = []
for name in dm.DATASETS:
    adata = sc.read_h5ad(dm.raw_dir(name) / "perturb_processed.h5ad")
    cond = adata.obs["condition"].astype(str)
    perts = [c for c in cond.unique() if c != "ctrl"]
    combos = [c for c in perts if "ctrl" not in c]
    rows.append({
        "dataset": name,
        "cells": adata.n_obs,
        "genes": adata.n_vars,
        "perturbations": len(perts),
        "two_gene_combinations": len(combos),
        "control_cells": int((cond == "ctrl").sum()),
        "median_cells_per_condition": int(cond.value_counts().median()),
    })
    del adata
pd.DataFrame(rows)

## Effect sizes and the noise floor

For each dataset: the distribution of `|condition mean - control mean|` across every (condition,
gene) pair, against the standard error of that same difference. A delta smaller than about two
standard errors is not distinguishable from no change at all.

The fraction below that line is the number that explains the direction-accuracy result in `04`.

In [ ]:
from pertreadout.readouts import fraction_below_noise_floor

summary = []
fig, axes = plt.subplots(1, len(dm.DATASETS), figsize=(4.2 * len(dm.DATASETS), 3.6))
for ax, name in zip(np.atleast_1d(axes), dm.DATASETS):
    adata = sc.read_h5ad(dm.prepared_dir(name, 32, 0) / "perturb_processed.h5ad")
    ctrl_mean, ctrl_var, n_ctrl = dm.control_statistics(adata)

    cond = adata.obs["condition"].astype(str).to_numpy()
    order = sorted(set(cond) - {"ctrl"})
    X = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
    means = np.stack([X[cond == c].mean(axis=0) for c in order])
    varis = np.stack([X[cond == c].var(axis=0) for c in order])
    counts = np.array([(cond == c).sum() for c in order])
    delta_se = np.sqrt(ctrl_var[None, :] / n_ctrl + varis / counts[:, None])

    delta = np.abs(means - ctrl_mean)
    frac = fraction_below_noise_floor(means, ctrl_mean, delta_se)
    summary.append({"dataset": name, "conditions": len(order),
                    "frac_delta_below_noise_floor": frac,
                    "median_abs_delta": float(np.median(delta)),
                    "p99_abs_delta": float(np.percentile(delta, 99))})

    ax.hist(delta.ravel(), bins=80, color="#3B6EA5")
    ax.axvline(np.median(1.96 * delta_se), color="black", ls="--", lw=1,
               label="median 1.96 x SE")
    ax.set_yscale("log")
    ax.set_title(name.capitalize()); ax.set_xlabel("|true delta|")
    ax.legend(frameon=False, fontsize=8)
    del adata, X
np.atleast_1d(axes)[0].set_ylabel("(condition, gene) pairs (log)")
fig.suptitle("Most of the mass sits at or below the measurement noise floor", fontsize=11)
fig.tight_layout(); plt.show()
pd.DataFrame(summary)

The y axis is log because the distribution is dominated by near-zero deltas; on a linear axis the
tail is invisible. That is stated rather than hidden — the honest-axis rule in this repository forbids
a *misleading* axis, not a log one, provided the caption says so.

## The split

`configs/*.yaml` use GEARS' `simulation` split at `train_gene_set_size: 0.75`, which holds out
perturbations whose genes were never seen during training. The alternative `no_test` — what the
source project used — holds out a random 10% of perturbations and, despite taking the argument,
ignores `train_gene_set_size` entirely.

In [ ]:
pert_data = dm.load_pert_data("dixit", seed=0, max_cells_per_condition=32, batch_size=16)
for split_name, conditions in pert_data.set2conditions.items():
    print(f"{split_name:6s} {len(conditions):4d} conditions   e.g. {conditions[:3]}")